# Notebook 32 — Tool Calling and Bounded Agent Loops

    ## Learning objectives

    - Define precise JSON tool contracts and validate arguments
- Implement a bounded model → tool → observation loop with HF inference
- Separate planning flexibility from authorization

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['huggingface-hub>=0.30,<1', 'python-dotenv>=1.1']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 32.1 Tools are capability boundaries

A tool schema is an interface contract, not a security boundary. Validate types and
values in code; enforce authentication and authorization outside the model; return
structured errors; set timeouts; make side effects idempotent where possible. A model's
decision to call a tool does not grant permission to perform consequential actions.


In [ ]:
import ast, json, operator
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def safe_calculator(expression: str) -> dict:
    def visit(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS: return OPS[type(node.op)](visit(node.left), visit(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS: return OPS[type(node.op)](visit(node.operand))
        raise ValueError("Unsupported expression")
    if len(expression) > 100: raise ValueError("Expression too long")
    return {"result": visit(ast.parse(expression, mode="eval").body)}

TOOLS = [{"type": "function", "function": {"name": "calculator",
    "description": "Evaluate arithmetic with numbers and +,-,*,/,** only.",
    "parameters": {"type": "object", "properties": {"expression": {"type": "string"}},
                   "required": ["expression"], "additionalProperties": False}}}]
print(safe_calculator("(12 + 3) * 4"))


In [ ]:
# Optional remote HF loop. Provider/model tool support varies.
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
load_dotenv()

def run_agent(question, max_steps=5):
    token = os.getenv("HUGGINGFACE_TOKEN")
    if not token: return "Set HUGGINGFACE_TOKEN to run the agent."
    client = InferenceClient(token=token)
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.chat_completion(model=os.getenv("HF_CHAT_MODEL", "Qwen/Qwen2.5-7B-Instruct-1M"),
            messages=messages, tools=TOOLS, tool_choice="auto", max_tokens=300)
        message = response.choices[0].message
        messages.append(message)
        if not message.tool_calls: return message.content
        for call in message.tool_calls:
            if call.function.name != "calculator": raise ValueError("Tool not allowed")
            args = json.loads(call.function.arguments)
            result = safe_calculator(**args)
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result)})
    return "Stopped: step budget exhausted"

print(run_agent("What is (17 * 23) + 9?"))


## 32.2 Reliability controls

Limit steps, wall time, tokens, tool calls, and spend. Detect repeated identical calls.
Distinguish read-only tools from reversible and irreversible actions. Require human
confirmation at policy boundaries. Persist a trace of decisions, validated arguments,
results, errors, and final output with sensitive fields redacted.


## 32.3 Parallel calls, cancellation, and mutation safety

Execute parallel calls only when they are independent, read-only or separately idempotent, and within a global
concurrency limit. Preserve tool-call IDs so results return to the correct requests. Cancel outstanding calls when
the client disconnects or the answer is complete, but remember that cancellation cannot undo an already committed
external effect. Use deadlines rather than unbounded per-call timeouts.

Mutating tools accept a host-generated idempotency key and authenticated actor context outside model-controlled
arguments. Split proposal from commit: the model proposes a typed action, policy validates it, a user may approve
its immutable digest, and an executor commits once and returns a receipt. A retry returns that receipt. Notebook 35
turns these rules into a durable workflow.


In [ ]:
import asyncio, hashlib
async def bounded_read(expression, semaphore):
    async with semaphore:
        await asyncio.sleep(0)
        return safe_calculator(expression)
semaphore = asyncio.Semaphore(2)
expressions = ["2+2", "17*23", "81/9"]
results = await asyncio.gather(*(bounded_read(value, semaphore) for value in expressions))
print(dict(zip(expressions, results)))
proposal = {"tool":"publish", "arguments":{"artifact":"report-7"}}
print("approval digest:", hashlib.sha256(json.dumps(proposal, sort_keys=True).encode()).hexdigest())


## 32.3 Tool-schema design in depth

Tool descriptions should state purpose, when to use/not use it, parameter semantics, units,
defaults, allowed ranges, and result/error shape. Prefer small orthogonal tools over one giant
“do anything” endpoint, but avoid dozens of nearly identical names. JSON Schema constrains
syntax; application validation enforces business rules, identity, ownership, quotas, and
cross-field invariants. Use enums and `additionalProperties: false` where supported.

Results should be compact structured data with stable fields. Distinguish success, retryable
error, permanent error, not-found, and authorization denied without exposing internal secrets.
Huge raw pages consume context and carry injection risk; tools should extract bounded relevant
fields and include provenance. Tool calls need correlation/idempotency keys for safe retries.
Never encode hidden authority in natural-language descriptions.


In [ ]:
# Validate tool arguments independently of what the model produced.
from pydantic import BaseModel, ConfigDict, Field, ValidationError
class CalculationArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    expression: str = Field(min_length=1, max_length=100)

for candidate in [{"expression": "2+2"}, {"expression": "x"*101},
                  {"expression": "2+2", "admin": True}]:
    try: print("valid", CalculationArgs.model_validate(candidate).model_dump())
    except ValidationError as exc: print("invalid", candidate, exc.errors()[0]["type"])


## 32.4 Agent loop states and control flow

A robust loop is a state machine: receive request → call model → validate proposed calls →
authorize → execute with timeout → normalize observation → append exact tool-call/result IDs →
repeat or finish. Handle multiple calls, partial failure, invalid JSON, unknown tools, model
refusal, context overflow, cancellation, and budgets. The model should see recoverable errors so
it can revise arguments, but repeated failures terminate deterministically.

Parallelize only independent, authorized, read-only calls. Calls whose inputs depend on prior
results remain sequential. Side effects should use a prepare/confirm/commit pattern: the model
drafts an action; code computes exact impact; the user approves that impact; code commits once.
Approval cannot be an instruction buried in the same untrusted context. Store approval scope and
expiry outside the model.


In [ ]:
# A deterministic loop-policy object independent of any model provider.
from dataclasses import dataclass, field
@dataclass
class AgentBudget:
    max_steps: int = 5
    max_tool_calls: int = 8
    calls: int = 0
    seen: set = field(default_factory=set)
    def admit(self, step, name, arguments):
        signature = (name, json.dumps(arguments, sort_keys=True))
        if step >= self.max_steps: return False, "step budget"
        if self.calls >= self.max_tool_calls: return False, "tool budget"
        if signature in self.seen: return False, "repeated call"
        self.seen.add(signature); self.calls += 1
        return True, "allowed"

budget = AgentBudget()
for step in range(3): print(step, budget.admit(step, "calculator", {"expression": "2+2"}))


## 32.5 Planning patterns and when not to use an agent

A direct tool call is best for known workflows. A deterministic DAG/state machine is best when
steps and transitions are known but tool results vary. An agent loop is justified when the next
action depends on open-ended observations and flexibility outweighs added latency, cost, and
risk. “Agent” does not require exposing hidden chain-of-thought; observable action/observation
traces and concise decision summaries are enough for operations.

Plan-and-execute separates a proposed plan from execution but plans become stale after new
observations. ReAct interleaves action and observation. Reflection/reviewer loops add cost and
can reinforce errors unless evaluated. Multi-agent systems multiply coordination and security
surfaces; use them only when roles have genuinely separable information/capabilities.

**Reference checklist:** bounded budgets; allowlisted tool registry; schema+semantic validation;
external authorization; timeouts/cancellation; idempotency; bounded observations; injection
handling; approval UX; trace/redaction; deterministic fallbacks; and trajectory-level evals.


## 32.6 Agent/tool reference

| Layer | Responsibility |
|---|---|
| Model | Propose calls/arguments and synthesize result |
| Loop/host | Maintain state, budgets, call/result correlation |
| Validator | Enforce JSON types and semantic constraints |
| Authorizer | Bind user identity, ownership, permission, approval |
| Tool service | Execute idempotently with timeout and scoped credentials |
| Evaluator/trace | Verify trajectory, effects, answer, cost, and safety |

Stopping conditions include final response, step/tool/token/deadline budget, repeated calls, nonretryable
error, cancellation, or approval denial. Parallelize only independent calls. Treat all tool results as
untrusted bounded data. Tool success does not mean task success; final eloquence does not mean calls
were safe/correct.

Evaluate tool selection, argument exactness, unnecessary calls, ordering/dependencies, error recovery,
side effects, citation/use of observations, final correctness, latency, and cost. Run deterministic
workflows without an agent when the state graph is known—the simplest sufficient controller is usually
more reliable and auditable.


## Exercises

    1. Add a typed lookup tool and tests for malformed arguments.
2. Detect repeated calls and stop with a diagnostic trace.
3. Create an evaluation set for tool selection, arguments, and final answers.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
